# PhiloMind — Post-Fix Evaluation

Verifica se il fix del data leakage ha funzionato:
- Train/test split senza contaminazione
- Loss curve train/val
- Accuracy reale sul test set
- Confronto col 99% artefatto di prima

In [ ]:
import sys, os, json, pickle
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')
%matplotlib inline

print('Ready')

## 1. Carica dati

Verifica che train/test siano stati generati senza leakage.

In [ ]:
train_df = pd.read_csv('../data/labels/questions_train.csv')
test_df = pd.read_csv('../data/labels/questions_test.csv')

print(f'Train: {len(train_df)} domande')
print(f'Test:  {len(test_df)} domande')
print(f'\nDistribuzione label (train):')
print(train_df['label'].value_counts())
print(f'\nDistribuzione label (test):')
print(test_df['label'].value_counts())

### 1.1 Verifica leakage

Controlla similarità tra train e test (prima era 19.4% > 0.8).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

all_qs = pd.concat([train_df['question'], test_df['question']])
vec = TfidfVectorizer().fit_transform(all_qs)
n_train = len(train_df)
sims = cosine_similarity(vec[:n_train], vec[n_train:])

max_sim = sims.max(axis=1)
high_sim = (max_sim > 0.8).sum()
print(f'Coppie train-test con similarita > 0.8: {high_sim} ({high_sim/n_train:.1%} del train)')
print(f'  Prima del fix: 19.4%')
print(f'  Dopo il fix:   {high_sim/n_train:.1%}')

if high_sim > 0:
    print('\n--- Coppie piu simili ---')
    for i in np.argsort(-max_sim)[:5]:
        j = sims[i].argmax()
        if max_sim[i] > 0.6:
            print(f'  Sim={max_sim[i]:.2%}')
            print(f'    TRAIN: {train_df.iloc[i]["question"][:100]}')
            print(f'    TEST:  {test_df.iloc[j]["question"][:100]}')

## 2. Carica modello

Se il modello non esiste, ritraina con `scripts/build_models.py`.

In [ ]:
from src.classification.bilstm import BiLSTMClassifier

model_dir = Path('../models/bilstm')
model_path = model_dir / 'final.pt'
vocab_path = model_dir / 'vocab.pkl'
label_path = model_dir / 'label2idx.json'
history_path = model_dir / 'training_history.json'

if model_path.exists():
    clf = BiLSTMClassifier(vocab_size=1)
    clf.load(str(model_path), str(vocab_path), str(label_path))
    print('Modello caricato')
else:
    print('Modello non trovato. Esegui prima: python scripts/build_models.py')
    raise SystemExit(1)

## 3. Loss curve

Se train loss scende e val loss risale = **overfitting**.
Se entrambe scendono e si stabilizzano = **OK**.

In [ ]:
if history_path.exists():
    h = json.load(open(history_path))
    
    fig, ax = plt.subplots(figsize=(10, 5))
    epochs = range(1, len(h['train_losses']) + 1)
    ax.plot(epochs, h['train_losses'], 'b-o', label='Train loss', markersize=4)
    ax.plot(epochs, h['val_losses'], 'r-s', label='Val loss', markersize=4)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training / Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Check overfitting
    overfit_epochs = []
    for i in range(1, len(h['train_losses'])):
        if h['val_losses'][i] > h['val_losses'][i-1] and h['train_losses'][i] < h['train_losses'][i-1]:
            overfit_epochs.append(i+1)
    
    if overfit_epochs:
        print(f'OVERFITTING sospetto alle epoche: {overfit_epochs}')
    else:
        print('Nessun segnale di overfitting nella loss curve.')
    
    print(f'\nTrain loss finale: {h["train_losses"][-1]:.4f}')
    print(f'Val loss finale:   {h["val_losses"][-1]:.4f}')
    print(f'Epoche totali: {len(h["train_losses"])} (early stopping a {h.get("stopped_epoch", "?")})')
else:
    print('training_history.json non trovato. Ritraina con build_models.py')

## 4. Evaluation

Accuracy su train e test.

In [ ]:
train_preds = [clf.predict(q)[0] for q in train_df['question']]
test_preds = [clf.predict(q)[0] for q in test_df['question']]

train_acc = accuracy_score(train_df['label'], train_preds)
test_acc = accuracy_score(test_df['label'], test_preds)
test_f1 = f1_score(test_df['label'], test_preds, average='weighted')

print('=' * 55)
print(f'  Train accuracy:  {train_acc:.2%}')
print(f'  Test accuracy:   {test_acc:.2%}')
print(f'  Test F1 (weighted): {test_f1:.2%}')
print(f'  Gap train-test:  {train_acc - test_acc:.2%}')
print('=' * 55)

if train_acc - test_acc > 0.15:
    print('OVERFITTING: gap > 15%, il modello memorizza而非 generalizza')
elif test_acc > 0.95:
    print('SOSPETTO: accuracy troppo alta, possibile leakage residuo')
else:
    print('Accuracy realistica. Confronta col 99% artefatto di prima.')

In [ ]:
print('Classification Report (test set):')
labels = sorted(test_df['label'].unique())
print(classification_report(test_df['label'], test_preds, labels=labels))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_df['label'], test_preds, labels=labels)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=labels, yticklabels=labels)
ax.set_title('Confusion Matrix (test set)')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()

## 5. Test su domande reali

Verifica qualitativa del pipeline completo (classificazione + quiz).

In [ ]:
from src.pipeline.core import PhiloMindPipeline

pl = PhiloMindPipeline(
    retriever_path='../models/retrieval/tfidf.pkl',
    config_path='../config/disciplines.json',
)

test_questions = [
    'What is Cartesian dualism?',
    'How do Plato and Aristotle differ?',
    'Give an example of a valid syllogism.',
    'What does Nietzsche mean by the eternal return?',
    'How does the allegory of the cave relate to knowledge?',
    'Quiz me on Hegel.',
]

for q in test_questions:
    out = pl.process(q, top_k=2)
    print('---' * 18)
    print(f'Q: {q}')
    print(f'Class: {out.classification.predicted_label} ({out.classification.confidence:.1%})')
    print(f'Quiz: {out.quiz["question"]}')
    for opt in out.quiz['options']:
        mark = ' <--' if opt == out.quiz['correct_answer'] else ''
        print(f'  {opt[:100]}{mark}')
    print()

## 6. Riepilogo

| Indicatore | Prima (leakage) | Dopo (fix) | Giudizio |
|---|---|---|---|
| Test accuracy | 99.06% | ? | |
| Gap train-test | 0.94% | ? | |
| Leakage > 0.8 | 19.4% | ? | |
| Loss val risale? | Sconosciuto | ? | |
| | | | |